<a href="https://colab.research.google.com/github/pritampadhy/researchstrategic/blob/main/MultiAgentCrewLAng.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Core dependencies
%pip install -U -q  langchain crewai

# Optional but recommended
%pip install -U -q streamlit gradio pyyaml


In [ ]:
import os
import json
from google import genai
from google.genai import types
from crewai import Task, Crew, Agent
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
class ProductionCodeExecutionAgent:
    """Production-grade agent utilizing official Google GenAI Client and CrewAI."""

    def __init__(self, api_key: str = None, model_name: str = "gemini-1.5-flash"):
        self.api_key = api_key or os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
        if not self.api_key:
            raise ValueError("GEMINI_API_KEY or GOOGLE_API_KEY environment variable must be set.")

        # 1. Official Google GenAI Core Client
        self.client = genai.Client(api_key=self.api_key)
        self.model_name = model_name

        # 2. CrewAI LLM configuration (Wraps Gemini for CrewAI agents)
        self.llm = ChatGoogleGenerativeAI(
            model=self.model_name,
            google_api_key=self.api_key,
            verbose=True,
            temperature=0.1
        )

        # Define Agents with explicit LLM assignment
        self.executor = Agent(
            role="Python Code Analyst",
            goal="Analyze the logic and execution of Python code snippets.",
            backstory="You are a senior developer specializing in code performance.",
            allow_delegation=False,
            verbose=True,
            llm=self.llm
        )

        self.validator = Agent(
            role="Security Auditor",
            goal="Ensure the code snippet does not contain malicious patterns.",
            backstory="You are a cybersecurity expert focusing on safe code execution.",
            allow_delegation=False,
            verbose=True,
            llm=self.llm
        )

    def execute_code_via_gemini(self, code: str, requirements: str = "") -> str:
        """Generates response via Google GenAI Core Client."""
        sys_instruction = (
            "You are a senior software engineer. Evaluate the Python code safely."
        )
        prompt = f"Requirements: {requirements}\n\nCode:\n```python\n{code}\n```"

        response = self.client.models.generate_content(
            model=self.model_name,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=sys_instruction,
                temperature=0.1,
            )
        )
        return response.text

    async def run_workflow(self, task_description: str, code: str = None) -> dict:
        print(f"🚀 Initializing Async CrewAI Workflow with model: {self.model_name}...")

        analysis_task = Task(
            description=task_description,
            expected_output="A structured analysis report.",
            agent=self.executor
        )

        crew = Crew(
            agents=[self.executor, self.validator],
            tasks=[analysis_task],
            verbose=True
        )

        print("--- Starting Async Crew Kickoff ---")
        crew_result = await crew.kickoff_async()

        gemini_result = None
        if code:
            print("--- Executing Gemini Analysis ---")
            gemini_result = self.execute_code_via_gemini(code)

        return {
            "task": task_description,
            "team_status": "Success",
            "crew_report": str(crew_result),
            "gemini_output": gemini_result
        }

In [ ]:
if __name__ == "__main__":
    import os
    # Setting the API key in the environment
    os.environ["GEMINI_API_KEY"] = ""

    agent_instance = ProductionCodeExecutionAgent()

    # We use await here because run_workflow is now async
    res = await agent_instance.run_workflow(
        task_description="Analyze the logic of the provided squared numbers list comprehension.",
        code="print([x**2 for x in range(5)])"
    )

    print("\nResult:\n", res.get("gemini_output"))

In [ ]:
import os

# The API key was already set in cell 3JOiXOwKf84k
# We will use it directly from the environment
if "GEMINI_API_KEY" not in os.environ:
    os.environ["GEMINI_API_KEY"] = ""

try:
    # Initialize with the correct key and model name
    agent = ProductionCodeExecutionAgent(api_key=os.environ.get("GEMINI_API_KEY"))
    result = agent.run_workflow(
        task_description="Analyze the efficiency of a list comprehension for squaring numbers.",
        code="print([x**2 for x in range(5)])"
    )

    print("\n--- Workflow Result ---")
    print(f"Task: {result['task']}")
    print(f"Status: {result['team_status']}")
    print(f"Gemini Output:\n{result['gemini_output']}")

except Exception as e:
    print(f"An error occurred during execution: {e}")

An error occurred during execution: name 'ChatGoogleGenerativeAI' is not defined
